# Moist Air Property Test

Functional tests for the moist-air property layer.

The notebook checks:

- availability of psychrometric functions,
- moist-air state generation from `T/RH/p`,
- moist-air state generation from `T/W/p`,
- high-temperature cases above water boiling temperature at given pressure,
- moist-air transport properties for outside-flow calculations,
- wet-process psychrometric limits.

Core units:

- temperature: K
- pressure: Pa
- relative humidity: fraction `0.0 ... 1.0`
- humidity ratio: `kg_water/kg_dry_air`
- displayed humidity ratio: `g_water/kg_dry_air`
- moist-air enthalpy: `J/kg_dry_air`
- moist-air transport `cp`: `J/(kg_moist_air*K)`

In [ ]:
from core.psychrometrics.psychrolib_adapter import (
    humidity_ratio_from_t_rh,
    humidity_ratio_from_g_per_kg_dry_air,
    humidity_ratio_to_g_per_kg_dry_air,
    relative_humidity_from_t_w,
    dew_point_from_t_rh,
    dew_point_from_t_w,
    moist_air_enthalpy_from_t_rh,
    moist_air_enthalpy_from_t_w,
    moist_air_density_from_t_rh,
    moist_air_density_from_t_w,
    saturation_humidity_ratio,
    saturation_humidity_ratio_is_defined,
    saturation_vapor_pressure,
    max_relative_humidity_at_t_p,
)

from core.psychrometrics.moist_air import (
    MoistAirState,
    moist_air_state_from_t_rh,
    moist_air_state_from_t_w,
    moist_air_state_from_t_w_g_per_kg_da,
    saturated_moist_air_state,
)

from core.properties.moist_air_transport import (
    MoistAirTransportProvider,
    moist_air_transport_props_from_state,
    moist_air_transport_result_from_state,
    moist_air_transport_result_from_t_rh,
)

from core.properties.adapters import to_outside_fluid_props

from core.psychrometrics.wet_process import (
    WetSurfaceProcessResult,
    condensable_water_per_kg_dry_air,
    enthalpy_drop_to_surface_saturation,
    humidity_ratio_drop_to_saturation,
    saturated_state_at_surface,
    wet_surface_process_limit,
)

# Basic reference state: 20 degC, 50% RH, 1 atm.
T_ref = 293.15
RH_ref = 0.50
p_ref = 101325.0

air_ref = moist_air_state_from_t_rh(T=T_ref, RH=RH_ref, p=p_ref)
transport_ref = moist_air_transport_result_from_state(air_ref)

assert isinstance(air_ref, MoistAirState)
assert 0.0070 < air_ref.W < 0.0076, air_ref.W
assert 281.5 < air_ref.T_dew < 283.2, air_ref.T_dew
assert 38000.0 < air_ref.h < 39000.0, air_ref.h
assert 1.19 < air_ref.rho < 1.21, air_ref.rho

assert transport_ref.props.rho > 0.0
assert transport_ref.props.mu > 0.0
assert transport_ref.props.k > 0.0
assert transport_ref.props.cp > 0.0

print("Moist-air import and smoke test passed.")

Moist-air import and smoke test passed.


In [ ]:
# Practical matrix: moist-air psychrometric and transport properties.

moist_air_cases = [
    {"case": "cold ventilation air", "T_C": 5.0, "RH": 0.40, "p_bar": 1.01325},
    {"case": "indoor reference", "T_C": 20.0, "RH": 0.50, "p_bar": 1.01325},
    {"case": "warm humid air", "T_C": 30.0, "RH": 0.70, "p_bar": 1.01325},
    {"case": "hot humid air", "T_C": 45.0, "RH": 0.80, "p_bar": 1.01325},
    {"case": "warm low-pressure air", "T_C": 60.0, "RH": 0.20, "p_bar": 0.95},
    {"case": "hot dilute moisture", "T_C": 80.0, "RH": 0.30, "p_bar": 1.01325},
    {"case": "cold saturated air", "T_C": 5.0, "RH": 1.00, "p_bar": 1.01325},
    {"case": "hot saturated air", "T_C": 80.0, "RH": 1.00, "p_bar": 1.01325},

    # Above boiling at approx. atmospheric pressure:
    # RH may still be mathematically usable only if RH * p_ws(T) < p.
    {"case": "over boiling check by RH", "T_C": 120.0, "RH": 0.30, "p_bar": 1.01325},
    {"case": "over boiling check by hi-RH", "T_C": 120.0, "RH": 0.80, "p_bar": 1.01325},

    # Preferred engineering input for hot gas mixtures:
    {"case": "over boiling by W", "T_C": 120.0, "W_g_per_kg_da": 120.0, "p_bar": 1.01325},

]

rows = []

for case in moist_air_cases:
    T = case["T_C"] + 273.15
    p = case["p_bar"] * 1.0e5

    warning_codes = []

    try:
        if "RH" in case:
            input_mode = "T/RH/p"
            air = moist_air_state_from_t_rh(
                T=T,
                RH=case["RH"],
                p=p,
            )

        elif "W_g_per_kg_da" in case:
            input_mode = "T/W_g_per_kg_da/p"
            air = moist_air_state_from_t_w_g_per_kg_da(
                T=T,
                W_g_per_kg_da=case["W_g_per_kg_da"],
                p=p,
            )

        else:
            raise ValueError("Case must define either RH or W_g_per_kg_da.")

        transport = moist_air_transport_result_from_state(air)

        for warning in transport.warnings:
            warning_codes.append(warning.code)

        if saturation_humidity_ratio_is_defined(T, p):
            W_sat = saturation_humidity_ratio(T, p)
            W_sat_g_per_kg_da = humidity_ratio_to_g_per_kg_dry_air(W_sat)
        else:
            W_sat_g_per_kg_da = math.nan
            warning_codes.append("SATURATION_STATE_NOT_DEFINED_AT_T_P")

        p_ws = saturation_vapor_pressure(T)
        RH_max = max_relative_humidity_at_t_p(T, p)

        row = {
            "case": case["case"],
            "input_mode": input_mode,
            "T_C": case["T_C"],
            "p_bar": case["p_bar"],
            "RH_input": case.get("RH", math.nan),
            "W_input_g_per_kg_da": case.get("W_g_per_kg_da", math.nan),
            "RH_calc": air.RH,
            "RH_max_at_T_p": RH_max,
            "p_ws_bar": p_ws / 1.0e5,
            "W_g_per_kg_da": humidity_ratio_to_g_per_kg_dry_air(air.W),
            "W_sat_g_per_kg_da": W_sat_g_per_kg_da,
            "T_dew_C": air.T_dew - 273.15,
            "h_kJ_per_kg_da": air.h / 1000.0,
            "rho_kg_m3": air.rho,
            "cp_J_kg_moist_K": transport.props.cp,
            "mu_Pa_s": transport.props.mu,
            "k_W_mK": transport.props.k,
            "Pr": transport.props.mu * transport.props.cp / transport.props.k,
            "warnings": ", ".join(sorted(set(warning_codes))),
            "error": "",
        }

    except Exception as exc:
        row = {
            "case": case["case"],
            "input_mode": "ERROR",
            "T_C": case["T_C"],
            "p_bar": case["p_bar"],
            "RH_input": case.get("RH", math.nan),
            "W_input_g_per_kg_da": case.get("W_g_per_kg_da", math.nan),
            "RH_calc": math.nan,
            "RH_max_at_T_p": math.nan,
            "p_ws_bar": math.nan,
            "W_g_per_kg_da": math.nan,
            "W_sat_g_per_kg_da": math.nan,
            "T_dew_C": math.nan,
            "h_kJ_per_kg_da": math.nan,
            "rho_kg_m3": math.nan,
            "cp_J_kg_moist_K": math.nan,
            "mu_Pa_s": math.nan,
            "k_W_mK": math.nan,
            "Pr": math.nan,
            "warnings": "",
            "error": str(exc),
        }

    rows.append(row)

moist_air_df = pd.DataFrame(rows)

moist_air_df

AssertionError: 

In [ ]:
# Practical matrix: surface-temperature limits for possible condensation.
#
# These are local psychrometric limits, not a heat-exchanger solution.
# They answer: "If air reached this surface temperature, how much water could
# theoretically condense per kg dry air and what would be the enthalpy drop?"

process_cases = [
    {
        "case": "indoor air, dry surface",
        "T_air_C": 20.0,
        "RH": 0.50,
        "p_bar": 1.01325,
        "T_surface_C": 12.0,
    },
    {
        "case": "indoor air, wet surface",
        "T_air_C": 20.0,
        "RH": 0.50,
        "p_bar": 1.01325,
        "T_surface_C": 7.0,
    },
    {
        "case": "warm humid air, wet surface",
        "T_air_C": 30.0,
        "RH": 0.70,
        "p_bar": 1.01325,
        "T_surface_C": 18.0,
    },
    {
        "case": "hot humid air, strong cooling",
        "T_air_C": 45.0,
        "RH": 0.80,
        "p_bar": 1.01325,
        "T_surface_C": 30.0,
    },
    {
        "case": "frost warning case",
        "T_air_C": 20.0,
        "RH": 0.50,
        "p_bar": 1.01325,
        "T_surface_C": -3.0,
    },
]

rows = []

for item in process_cases:
    air = moist_air_state_from_t_rh(
        T=item["T_air_C"] + 273.15,
        RH=item["RH"],
        p=item["p_bar"] * 1.0e5,
    )

    limit = wet_surface_process_limit(
        air=air,
        T_surface=item["T_surface_C"] + 273.15,
    )

    assert isinstance(limit, WetSurfaceProcessResult)
    assert limit.condensable_water >= 0.0
    assert limit.enthalpy_drop >= 0.0

    # Public helper consistency checks.
    W_drop = humidity_ratio_drop_to_saturation(
        air=air,
        T_surface=item["T_surface_C"] + 273.15,
    )
    m_cond = condensable_water_per_kg_dry_air(
        air=air,
        T_surface=item["T_surface_C"] + 273.15,
    )
    dh = enthalpy_drop_to_surface_saturation(
        air=air,
        T_surface=item["T_surface_C"] + 273.15,
    )

    assert abs(W_drop - m_cond) < 1e-15
    assert abs(dh - limit.enthalpy_drop) < 1e-9

    rows.append(
        {
            "case": item["case"],
            "T_air_C": round(item["T_air_C"], 2),
            "RH_%": round(item["RH"] * 100.0, 1),
            "T_dew_C": round(air.T_dew - 273.15, 3),
            "T_surface_C": round(item["T_surface_C"], 2),
            "will_condense": limit.condensation.will_condense,
            "W_bulk_g_kg_da": round(air.W * 1000.0, 4),
            "W_surface_sat_g_kg_da": round(limit.surface_saturated_state.W * 1000.0, 4),
            "condensable_g_kg_da": round(limit.condensable_water * 1000.0, 4),
            "enthalpy_drop_kJ_kg_da": round(limit.enthalpy_drop / 1000.0, 3),
            "warnings": ", ".join(w.code for w in limit.warnings) or "-",
        }
    )

display_table(rows)
print("Wet-process limit matrix test passed.")

In [ ]:
# Practical provider check: fixed humidity ratio provider over a temperature range.
#
# This is useful for dry-side sensible calculations where the gas composition
# is fixed over the evaluated temperature range.

provider = MoistAirTransportProvider.from_t_rh(
    T=293.15,
    RH=0.50,
    p=101325.0,
)

rows = []

for T_C in [10.0, 20.0, 30.0, 40.0]:
    props = provider.at(T=T_C + 273.15, p=101325.0)
    Pr = props.mu * props.cp / props.k

    assert props.rho > 0.0
    assert props.mu > 0.0
    assert props.k > 0.0
    assert props.cp > 0.0

    rows.append(
        {
            "T_C": T_C,
            "rho_kg_m3": round(props.rho, 5),
            "mu_uPa_s": round(props.mu * 1.0e6, 4),
            "k_W_mK": round(props.k, 5),
            "cp_J_kg_maK": round(props.cp, 3),
            "Pr": round(Pr, 5),
        }
    )

display_table(rows)
print("MoistAirTransportProvider fixed-W matrix test passed.")